# Quickstart with Milvus Lite

向量是神经网络模型的输出数据格式，能够有效编码信息，在知识库、语义搜索、检索增强生成（RAG）等人工智能应用中发挥关键作用。
Milvus 是一个开源的向量数据库，适用于各种规模的人工智能应用，无论是运行 Jupyter 笔记本中的演示聊天机器人，还是构建可服务数十亿用户的网络级搜索系统。在本指南中，我们将带您快速完成 Milvus 在本地的部署，并使用 Python 客户端库生成、存储和查询向量。

In [1]:
#!uv pip install pymilvus

In [2]:
import sys
import os

print(sys.executable)
os.environ['HF_HOME'] = r'F:\Teewon\Milvue\models'
os.environ['HF_HOME'] = os.path.join(os.getcwd(), '../models')

F:\Teewon\Milvue\.venv\Scripts\python.exe


## Set Up Vector Database

设置向量数据库

In [3]:
#!uv pip install milvus-lite

In [4]:
from pymilvus import MilvusClient

client = MilvusClient("milvus_demo.db")

## Create a Collection(表)

在 Milvus 中，我们需要一个集合来存储向量及其关联的元数据。你可以将其想象为传统 SQL 数据库中的一个表。创建集合时，可以定义模式和索引参数，以配置向量的规格，例如维度、索引类型和距离度量。此外，还有复杂的概念用于优化索引，以提升向量搜索性能。目前，我们先专注于基本功能，尽可能使用默认值。至少，你只需要设置集合名称以及集合中向量字段的维度即可。

In [5]:
if client.has_collection(collection_name="demo_collection"):
    client.drop_collection(collection_name="demo_collection")
client.create_collection(
    collection_name="demo_collection",
    dimension=768,  # The vectors we will use in this demo has 768 dimensions
)

在上述设置中，
- 主键和向量字段使用其默认名称（“id”和“vector”）。
- 度量类型（向量距离定义）设置为默认值（COSINE）。
- 主键字段接受整数，且不会自动递增（即不使用自动ID功能）。您也可以按照以下说明，正式定义集合的模式。

## Prepare Data

在本指南中，我们使用向量对文本进行语义搜索。需要通过下载嵌入模型来生成文本的向量。这可以通过 pymilvus[model] 库中的实用函数轻松完成。

### Represent text with vectors

首先，安装模型库。该包包含 PyTorch 等必要的机器学习工具。如果您的本地环境从未安装过 PyTorch，下载该包可能需要一些时间。

In [6]:
# !uv pip install "pymilvus[model]"

使用默认模型生成向量嵌入。Milvus 期望数据以字典列表的形式插入，其中每个字典表示一条数据记录，称为实体。

In [7]:
from pymilvus.model.dense import SentenceTransformerEmbeddingFunction
# 如果连接到 https://huggingface.co/ 失败，请取消注释以下路径：
# import os
# os.environ['HF_ENDPOINT'] = 'https://hf-mirror.com'

# 这将下载一个小型嵌入模型“paraphrase-albert-small-v2”（约50MB）。
embedding_fn=SentenceTransformerEmbeddingFunction(
    model_name="all-mpnet-base-v2"
)

# 要搜索的文本字符串
docs = [
    "Artificial intelligence was founded as an academic discipline in 1956.",
    "Alan Turing was the first person to conduct substantial research in AI.",
    "Born in Maida Vale, London, Turing was raised in southern England.",
]

vectors=embedding_fn.encode_documents(docs)
# 输出向量有768个维度，与我们刚刚创建的Collection(表)相匹配
print("Dim:", embedding_fn.dim, vectors[0].shape)# Dim: 768 (768,)

# 每个实体都有ID、向量表示、原始文本以及一个我们用于后续演示元数据过滤的主体标签
data = [
    {"id": i, "vector": vectors[i], "text": docs[i], "subject": "history"}
    for i in range(len(vectors))
]

print("Data has", len(data), "entities, each with fields: ", data[0].keys())
print("Vector dim:", len(data[0]["vector"]))

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Dim: 768 (768,)
Data has 3 entities, each with fields:  dict_keys(['id', 'vector', 'text', 'subject'])
Vector dim: 768


F:\Teewon\Milvue\.venv\Lib\site-packages\pymilvus\model\dense\sentence_transformer.py:46: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  return self.model.get_sentence_embedding_dimension()


### [Alternatively] 使用随机向量的虚假表示

如果由于网络问题无法下载模型，作为替代方案，你可以使用随机向量来表示文本，仍然可以完成示例。但请注意，搜索结果不会反映语义相似性，因为这些向量是虚拟的。

In [ ]:
# import random
#
# # Text strings to search from.
# docs = [
#     "Artificial intelligence was founded as an academic discipline in 1956.",
#     "Alan Turing was the first person to conduct substantial research in AI.",
#     "Born in Maida Vale, London, Turing was raised in southern England.",
# ]
# # Use fake representation with random vectors (768 dimension).
# vectors = [[random.uniform(-1, 1) for _ in range(768)] for _ in docs]
# data = [
#     {"id": i, "vector": vectors[i], "text": docs[i], "subject": "history"}
#     for i in range(len(vectors))
# ]
#
# print("Data has", len(data), "entities, each with fields: ", data[0].keys())
# print("Vector dim:", len(data[0]["vector"]))

## Insert Data

让我们将数据插入到Collection中：

In [9]:
res=client.insert(collection_name="demo_collection",data=data)

print(res)

{'insert_count': 3, 'ids': [0, 1, 2]}


## Semantic Search语义搜索

现在我们可以将搜索查询文本表示为向量，并在 Milvus 上进行向量相似度搜索。

### Vector search

Milvus 可以同时接受一个或多个向量搜索请求。query_vectors 变量的值是一个向量列表，每个向量都是浮点数数组。

In [11]:
query_vectors=embedding_fn.encode_documents(["Who is Alan Turing?"])
# 如果你没有嵌入函数，可以使用一个虚拟向量来完成演示：
# query_vectors = [ [ random.uniform(-1, 1) for _ in range(768) ] ]

res=client.search(
    collection_name="demo_collection",  # target collection
    data=query_vectors, # query vectors
    limit=2,    # number of returned entities
    output_fields=["text","subject"]    # specifies fields to be returned
)

print(res)

data: [[{'id': 1, 'distance': 0.6916976571083069, 'entity': {'id': 1, 'text': 'Alan Turing was the first person to conduct substantial research in AI.', 'subject': 'history'}}, {'id': 2, 'distance': 0.5550014972686768, 'entity': {'id': 2, 'text': 'Born in Maida Vale, London, Turing was raised in southern England.', 'subject': 'history'}}]]


输出是一个结果列表，每个结果对应一个向量搜索查询。每个查询包含一组结果，每组结果包括实体主键、与查询向量的距离，以及带有指定输出字段的实体详细信息。

## Vector Search with Metadata Filtering

您还可以在考虑元数据值（在Milvus中称为“标量”字段，因为标量指的是非向量数据）的同时进行向量搜索。这通过指定特定条件的过滤表达式来实现。下面以主题字段为例，说明如何进行搜索和筛选。

In [12]:
# Insert more docs in another subject
docs=[
    "Machine learning has been used for drug design.",
    "Computational synthesis with AI algorithms predicts molecular properties.",
    "DDR1 is involved in cancers and fibrosis.",
]
vectors=embedding_fn.encode_documents(docs)
data=[
    {"id": 3 + i, "vector": vectors[i], "text": docs[i], "subject": "biology"}
    for i in range(len(vectors))
]

client.insert(collection_name="demo_collection",data=data)

# This will exclude any text in "history" subject despite close to the query vector.
res=client.search(
    collection_name="demo_collection",
    data=embedding_fn.encode_queries(["tell me AI related information"]),
    filter="subject=='biology'",
    limit=2,
    output_fields=["text","subject"])

print(res)

data: [[{'id': 4, 'distance': 0.2796586751937866, 'entity': {'id': 4, 'text': 'Computational synthesis with AI algorithms predicts molecular properties.', 'subject': 'biology'}}, {'id': 3, 'distance': 0.2689781188964844, 'entity': {'id': 3, 'text': 'Machine learning has been used for drug design.', 'subject': 'biology'}}]]


默认情况下，标量字段未被索引。如果需要在大数据集中进行元数据过滤搜索，可以考虑使用固定模式，并启用索引以提高搜索性能。

除了向量搜索外，您还可以进行其他类型的搜索：

### Query

query() 是一种操作，用于检索所有符合特定条件的实体，例如过滤表达式或匹配某些 ID。

例如，获取所有标量字段具有特定值的实体：

In [19]:
res=client.query(
    collection_name="demo_collection",
    filter="subject=='history'",
    output_fields=["text","subject"],
)
res


data: ["{'id': 0, 'text': 'Artificial intelligence was founded as an academic discipline in 1956.', 'subject': 'history'}", "{'id': 1, 'text': 'Alan Turing was the first person to conduct substantial research in AI.', 'subject': 'history'}", "{'id': 2, 'text': 'Born in Maida Vale, London, Turing was raised in southern England.', 'subject': 'history'}"], extra_info: {}

通过主键primary key直接获取实体：

In [18]:
res=client.query(
    collection_name="demo_collection",
    ids=[0,2],
    output_fields=["vector","text","subject"],
)
res

data: ["{'id': 0, 'vector': [0.0009003584855236113, 0.037085775285959244, -0.048853643238544464, -0.00790131464600563, -0.02275817096233368, 0.006343542598187923, 0.09461662173271179, 0.007052363362163305, 0.02285480685532093, -0.02145242504775524, 0.046696774661540985, 0.05948086082935333, -0.01994618959724903, 0.02788143791258335, 0.019986707717180252, 0.034199927002191544, -0.001127908588387072, 0.02888294868171215, 0.05905904993414879, -0.0013345909537747502, -0.0480647012591362, -0.02459084428846836, -0.05375463142991066, -0.013748066499829292, -0.005146491806954145, -0.013935085386037827, -0.03143699839711189, -0.05096196010708809, -0.0005650188541039824, -0.015357050113379955, 0.02119762822985649, -0.036535970866680145, -0.011677077040076256, -0.010798638686537743, 1.2127479749324266e-06, -0.011674565263092518, 0.012950638309121132, 0.03800461068749428, -0.037896528840065, -0.01330237090587616, 0.011308995075523853, 0.08604826778173447, 0.006760702934116125, -0.02157496847212314

### client.query和client.search的区别

query 是精确查找，search 是相似度匹配

## Delete Entities

如需清除数据，您可以删除指定主键的实体，或删除符合特定筛选条件的所有实体。

返回list(id)

In [20]:
# Delete entities by primary key
res=client.delete(collection_name="demo_collection", ids=[0,2])

print(res)

# Delete entities by a filter expression
res=client.delete(
    collection_name="demo_collection",
    filter="subject=='biology'",
)

print(res)

[0, 2]
[3, 4, 5]


## Load Existing Data

由于 Milvus Lite 的所有数据都存储在本地文件中，即使程序终止后，您仍可通过使用现有文件创建一个 MilvusClient 来将所有数据加载到内存中。例如，这将从“milvus_demo.db”文件中恢复集合，并继续向其中写入数据。

In [ ]:
from pymilvus import MilvusClient

client = MilvusClient("milvus_demo.db")

## Drop the collection

如果你想删除集合中的所有数据，可以使用以下命令删除该集合：

In [21]:
# Drop collections
client.drop_collection(collection_name="demo_collection")

了解更多 Milvus Lite 适合用于本地 Python 程序的入门。如果您拥有大规模数据，或希望在生产环境中使用 Milvus，可以了解如何在 Docker 和 Kubernetes 上部署 Milvus。Milvus 的所有部署模式都共享相同的 API，因此在切换到其他部署模式时，您的客户端代码几乎无需更改。只需指定任意位置部署的 Milvus 服务器的 URI 和 Token 即可：

In [ ]:
client = MilvusClient(uri="http://localhost:19530", token="root:Milvus")

Milvus 提供 REST 和 gRPC API，并提供 Python、Java、Go、C# 和 Node.js 等语言的客户端库。